# Simple Neural Network With PyTorch Lightning

In this lab, we build with PyTorch Lightning neural networks that recognize MNIST's handwritten digits. Lightning writes the training loop for us: the model stays a PyTorch `nn.Module`, and a `Trainer` trains it (slides "PyTorch Lightning", "Defining a `LightningModule`" and "Training With the `Trainer`").

1. a multinomial logistic regression, the network without a hidden layer, with two ways of giving the classes to the loss, then how its output must match its loss;
2. a regularized multilayer perceptron;
3. monitoring training with TensorBoard, to compare two batch sizes;
4. a hyperparameter search with Optuna.

You don't need a GPU for this lab: each training is fast on CPU, and only the hyperparameter search, the last part, takes a few minutes. You can therefore switch runtimes if the one you were given has a GPU (`Runtime > Change runtime type`).

In [ ]:
!pip install -q lightning torchmetrics optuna tensorboard
import pathlib

import lightning
import matplotlib.pyplot as plt
import optuna
import pandas
import torch
import torchmetrics
import torchvision
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

# Same random draws at each run: reproducible results
lightning.seed_everything(0)

## Fetching and Preparing the Data

As in the previous lab, torchvision provides the MNIST dataset: 60,000 training images and 10,000 test images, handwritten digits of 28 × 28 grayscale pixels, with their label, from 0 to 9. The following cell loads them and prepares them the same way: each image flattened into a vector of 784 pixels, the input of an `nn.Linear` layer, and its pixels brought from $[0, 255]$ to $[0, 1]$.

In [ ]:
nb_classes = 10
input_dim = 28 * 28

train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
# "Flat" images of input_dim pixels in [0, 1]: -1 lets PyTorch compute the
# number of examples
X_train = train_data.data.reshape(-1, input_dim) / 255
X_test = test_data.data.reshape(-1, input_dim) / 255
y_train, y_test = train_data.targets, test_data.targets
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

## One-Hot Encoding of the Classes

The labels `y_train` and `y_test` are class indices, integers from 0 to 9. Encode them as [*one-hot*](https://en.wikipedia.org/wiki/One-hot) vectors: for each example, 10 values, all zero except a 1 at the index of its class. This is the first of the two ways of giving the classes to cross-entropy we'll see.

Store the encoded classes in `Y_train` and `Y_test`, and keep `y_train` and `y_test`: what follows uses both.

**Useful documentation**

- [`torch.nn.functional.one_hot`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `nn.functional.one_hot(y_train, nb_classes)` returns an integer tensor of shape `(60000, 10)`; `.float()` converts it to floats, the type cross-entropy expects for one-hot targets. Label 5, the first example's, becomes `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
Y_train = nn.functional.one_hot(..., nb_classes).float()
Y_test = ...
```

### Solution

In [ ]:
# The classes as one-hot vectors, as floats for cross-entropy
Y_train = nn.functional.one_hot(y_train, nb_classes).float()
Y_test = nn.functional.one_hot(y_test, nb_classes).float()
print(Y_train.shape, Y_test.shape)
print(f"Label {int(y_train[0])} as one-hot: {Y_train[0].tolist()}")

## The Lab's `LightningModule`

The following cell defines the tools of the next parts:

- `MNISTClassifier`, the lab's `LightningModule` (slide "Defining a `LightningModule`"): it wraps a PyTorch model that returns *logits*, the raw scores of the 10 classes, and takes care of the loss, the accuracy (with torchmetrics) and the optimizer. The loss is cross-entropy, which applies the softmax itself (slide "Classic Pitfall: Softmax and Cross-Entropy") and accepts one-hot vectors as targets as well as class indices. The optimizer, its learning rate and its weight decay are parameters;
- `get_dataloaders`, which splits data into training and validation (20% by default) and returns their `DataLoader`s;
- `evaluate`, which displays a trained model's loss and accuracy on the test set;
- `plot_losses`, which plots, epoch by epoch, the training and validation losses of a training run, from the `metrics.csv` file its `CSVLogger` writes into `trainer.logger.log_dir`.

In [ ]:
class MNISTClassifier(lightning.LightningModule):
  """Lightning wrapper: cross-entropy on the logits, accuracy as a metric."""

  def __init__(
    self,
    model: nn.Module,
    optimizer: type[torch.optim.Optimizer] = torch.optim.SGD,
    learning_rate: float = 0.01,
    weight_decay: float = 0.0,
    num_classes: int = nb_classes,
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # An example input lets Lightning display the shape of each layer's input
    # and output tensors in the model summary
    self.example_input_array = torch.zeros(1, input_dim)
    # torchmetrics needs one metric instance per stage
    self.accuracies = nn.ModuleDict(
      {
        f"{stage}_accuracy": torchmetrics.Accuracy(
          task="multiclass", num_classes=num_classes
        )
        for stage in ("train", "val", "test")
      }
    )

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    images, targets = batch
    logits = self(images)
    loss = nn.functional.cross_entropy(logits, targets)
    # The metric, for its part, needs class indices
    labels = targets.argmax(dim=-1) if targets.dim() > 1 else targets
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, labels)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "test")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return self.hparams.optimizer(
      self.parameters(),
      lr=self.hparams.learning_rate,
      weight_decay=self.hparams.weight_decay,
    )


def get_dataloaders(
  X: torch.Tensor, Y: torch.Tensor, batch_size: int = 128, validation_split: float = 0.2
) -> tuple[DataLoader, DataLoader]:
  """Split the data between training and validation."""
  train_dataset, val_dataset = random_split(
    TensorDataset(X, Y), [1 - validation_split, validation_split]
  )
  return (
    DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
    DataLoader(val_dataset, batch_size=batch_size),
  )


def evaluate(
  trainer: lightning.Trainer, classifier: MNISTClassifier, one_hot: bool = False
) -> None:
  """Display the loss and the accuracy on the test set.

  one_hot: True for a model trained on the one-hot targets (Y_train).
  """
  test_loader = DataLoader(
    TensorDataset(X_test, Y_test if one_hot else y_test), batch_size=128
  )
  score = trainer.test(classifier, test_loader, verbose=False)[0]
  print(f"Test: loss {score['test_loss']:.3f}, accuracy {score['test_accuracy']:.3f}")


def plot_losses(trainer: lightning.Trainer) -> None:
  """Plot the training and validation losses a CSVLogger has written."""
  # The training and validation metrics aren't written on the same lines of the
  # CSV file: we group them by epoch
  metrics = (
    pandas.read_csv(pathlib.Path(trainer.logger.log_dir) / "metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
    .dropna(subset=["train_loss", "val_loss"])
  )
  plt.plot(metrics.index, metrics["train_loss"], label="Training")
  plt.plot(metrics.index, metrics["val_loss"], label="Validation")
  plt.xlabel("Epoch")
  plt.ylabel("Loss")
  plt.legend()
  plt.show()

## Multinomial Logistic Regression

Create `model`, a network without a hidden layer: each of its `nb_classes` outputs is a linear combination of the `input_dim` pixels, a logit, and the cross-entropy's softmax turns them into 10 probabilities summing to 1. This is a multinomial logistic regression.

Wrap it in `classifier`, an `MNISTClassifier` with its default settings. Display its summary with `ModelSummary`, and explain its number of parameters.

Then train `classifier` with a `lightning.Trainer` (slide "Training With the `Trainer`"):

- batches of 128 examples, with 20% of the training data for validation: `get_dataloaders` on `X_train` and the one-hot targets `Y_train`;
- 10 epochs;
- a `CSVLogger`, which records the metrics in `logs/simple`, and no checkpoint saving.

Plot its loss curves with `plot_losses`: is the validation loss still going down at the last epoch? Finally, measure its loss and accuracy on the test set with `evaluate`. Keep these figures: they are the baseline for the next models.

**Useful documentation**

- [`lightning.Trainer`](https://lightning.ai/docs/pytorch/stable/common/trainer.html)
- [`CSVLogger`](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.CSVLogger.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `nn.Sequential(nn.Linear(input_dim, nb_classes))`
- `MNISTClassifier(model)`: SGD, with a learning rate of 0.01
- `print(ModelSummary(classifier, max_depth=-1))`: `max_depth=-1` details all the layers, including those of the wrapped model
- `get_dataloaders(X_train, Y_train, batch_size=128)` returns the training and validation `DataLoader`s
- `lightning.Trainer(max_epochs=10, accelerator="auto", devices=1, logger=CSVLogger("logs", name="simple"), enable_checkpointing=False)`: `accelerator="auto"` takes the GPU if there is one
- `callbacks=[TQDMProgressBar()]`: tqdm's progress bar, rather than rich's, which Lightning picks when rich is installed (Colab) and which froze some trainings
- `trainer.fit(classifier, train_loader, val_loader)`, then `plot_losses(trainer)`
- `evaluate(trainer, classifier, one_hot=True)`: the model learns on one-hot targets
- Expected result: 7,850 parameters; about 89% accuracy on validation as on test; SGD, with a learning rate of 0.01, is still improving at the tenth epoch

A skeleton to copy into the exercise cell, replacing each `...`:

```python
model = nn.Sequential(nn.Linear(..., ...))
classifier = MNISTClassifier(...)
print(ModelSummary(..., max_depth=-1))

train_loader, val_loader = get_dataloaders(..., ..., batch_size=...)
trainer = lightning.Trainer(
  max_epochs=...,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name=...),
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer.fit(..., ..., ...)
plot_losses(...)
evaluate(trainer, classifier, one_hot=...)
```

### Solution

In [ ]:
model = nn.Sequential(nn.Linear(input_dim, nb_classes))
classifier = MNISTClassifier(model)
print(ModelSummary(classifier, max_depth=-1))

# A linear layer has one weight per input and per output, plus one bias per
# output: 784 × 10 + 10 = 7,850 parameters

In [ ]:
# The Trainer runs the LightningModule's complete training loop
train_loader, val_loader = get_dataloaders(X_train, Y_train, batch_size=128)

trainer = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="simple"),
  # tqdm's progress bar, rather than rich's, which Lightning picks when rich
  # is installed (Colab): it froze some trainings
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer.fit(classifier, train_loader, val_loader)
# The validation loss is still going down at the tenth epoch: with a learning
# rate of 0.01, SGD hasn't finished converging
plot_losses(trainer)
evaluate(trainer, classifier, one_hot=True)

## Matching the Output and the Loss

PyTorch's cross-entropy (`nn.functional.cross_entropy`, in `MNISTClassifier`) also accepts the integer labels `y_train` as targets (class indices, of type `torch.int64`): one-hot encoding becomes unnecessary, and the next models do without it.

It always expects logits, and applies the softmax itself. A model ending with a softmax therefore gives it probabilities, which it reads as logits: the wrong pair.

Train on `y_train`, with the `torch.optim.Adam` optimizer, a learning rate of 10⁻³ and the settings of the previous part, two logistic regressions:

1. `adam_classifier`, wrapping a new model identical to `model` (`CSVLogger` in `logs/adam`);
2. `softmax_classifier`, whose model ends with an `nn.Softmax` layer (`CSVLogger` in `logs/softmax`).

Compare their test loss and accuracy: which one has a wrong loss? Does it learn worse? Why does PyTorch report no error?

**Useful documentation**

- [`torch.nn.Softmax`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Softmax.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `nn.Sequential(nn.Linear(input_dim, nb_classes), nn.Softmax(dim=-1))`: the softmax over the last dimension, the classes'
- `MNISTClassifier(model, optimizer=torch.optim.Adam, learning_rate=1e-3)`
- `get_dataloaders(X_train, y_train, batch_size=128)`, a new `Trainer` per model, then `evaluate(trainer, classifier)`, without `one_hot`
- A function `train(model, name)` that wraps, trains and evaluates a model avoids writing the same code twice
- Expected result: about 92% test accuracy for `adam_classifier`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def train(model: nn.Module, name: str) -> MNISTClassifier:
  classifier = MNISTClassifier(model, optimizer=..., learning_rate=...)
  train_loader, val_loader = get_dataloaders(X_train, ..., batch_size=128)
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name=name),
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifier, train_loader, val_loader)
  evaluate(trainer, classifier)
  return classifier


adam_classifier = train(nn.Sequential(nn.Linear(input_dim, nb_classes)), "adam")
softmax_classifier = train(nn.Sequential(..., ...), "softmax")
```

### Solution

In [ ]:
def train(model: nn.Module, name: str) -> MNISTClassifier:
  """Train and evaluate a model with Adam, on the integer labels y_train."""
  classifier = MNISTClassifier(model, optimizer=torch.optim.Adam, learning_rate=1e-3)
  train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=128)
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name=name),
    # tqdm's progress bar, rather than rich's, which Lightning picks when rich
    # is installed (Colab): it froze some trainings
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifier, train_loader, val_loader)
  evaluate(trainer, classifier)
  return classifier


# Right pair: the model returns logits, the cross-entropy applies the softmax
adam_classifier = train(nn.Sequential(nn.Linear(input_dim, nb_classes)), "adam")
# Wrong pair: one softmax too many, before the cross-entropy's own
softmax_classifier = train(
  nn.Sequential(nn.Linear(input_dim, nb_classes), nn.Softmax(dim=-1)), "softmax"
)

# Close accuracies, about 92.5% on the test set, 3 points more than with SGD with
# the same architecture, but a wrong loss for softmax_classifier: about 1.55 instead
# of 0.27. The cross-entropy's softmax is applied to probabilities, between 0 and 1:
# the probability it gives to the right class never exceeds e / (e + 9), that is 23%,
# and the loss never goes below 1.46. The gradients are squashed; Adam, which adapts
# the size of its steps, makes up for it here, but with SGD the same regression
# learns much more slowly (about 79% against 89% after 10 epochs). PyTorch sees no
# error: probabilities have the shape and type of logits.

## Regularized Multilayer Perceptron

Write a `build_deep_model` function that returns a fresh multilayer perceptron:

- 4 hidden layers of 20 neurons, with the ReLU activation;
- for all the layers, He initialization, suited to ReLU (`nn.init.kaiming_normal_`, slide "Weight Initialization");
- an output layer of `nb_classes` neurons, without activation: it returns logits, like the logistic regression.

Wrap one in `deep_classifier`, an `MNISTClassifier` training it with the `torch.optim.AdamW` optimizer, a learning rate of 10⁻³ and a weight decay of 0.01: the regularization is carried by the optimizer, which shrinks the weights at each step to curb overfitting (slide "AdamW and Weight Regularization"). Display its summary, then train it on the integer labels with the same settings as the logistic regression. How many points of test accuracy does it gain over `adam_classifier`, the logistic regression trained with Adam? For how many more parameters? Also compare the training and validation accuracies of the last epoch: does this network overfit?

The next parts reuse `build_deep_model`.

**Useful documentation**

- [`torch.nn.init.kaiming_normal_`](https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.kaiming_normal_)

In [ ]:
def build_deep_model() -> nn.Module: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")` replaces in place the weights of an `nn.Linear` layer with draws from a normal distribution of variance 2 / n_in, which keep the variance of the signal from one ReLU layer to the next; PyTorch's default initialization, uniform, isn't designed for ReLU
- A `dense(in_features, out_features)` function creating the layer and initializing its weights avoids repeating these two lines
- `nn.Sequential(dense(input_dim, 20), nn.ReLU(), ...)`: each hidden layer is followed by its activation
- `MNISTClassifier(build_deep_model(), optimizer=torch.optim.AdamW, learning_rate=1e-3, weight_decay=0.01)`
- `deep_trainer.callback_metrics["train_accuracy"]` and `["val_accuracy"]`: the training and validation accuracies of the last epoch, to read before `evaluate`, whose test replaces them
- Expected result: 17,170 parameters, about 95.5% test accuracy

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def dense(in_features: int, out_features: int) -> nn.Linear:
  layer = nn.Linear(in_features, out_features)
  nn.init.kaiming_normal_(..., nonlinearity=...)
  return layer


def build_deep_model() -> nn.Module:
  return nn.Sequential(
    dense(input_dim, 20),
    nn.ReLU(),
    ...,
    dense(20, nb_classes),
  )


deep_classifier = MNISTClassifier(
  build_deep_model(), optimizer=..., learning_rate=..., weight_decay=...
)
print(ModelSummary(deep_classifier, max_depth=-1))
train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=128)
deep_trainer = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="deep"),
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
deep_trainer.fit(..., ..., ...)
print(deep_trainer.callback_metrics[...], deep_trainer.callback_metrics[...])
evaluate(deep_trainer, deep_classifier)
```

### Solution

In [ ]:
def dense(in_features: int, out_features: int) -> nn.Linear:
  """A linear layer whose weights get He initialization, for ReLU."""
  layer = nn.Linear(in_features, out_features)
  nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
  return layer


def build_deep_model() -> nn.Module:
  """A fresh multilayer perceptron: 4 hidden layers of 20 neurons."""
  return nn.Sequential(
    dense(input_dim, 20),
    nn.ReLU(),
    dense(20, 20),
    nn.ReLU(),
    dense(20, 20),
    nn.ReLU(),
    dense(20, 20),
    nn.ReLU(),
    dense(20, nb_classes),
  )


# The regularization is carried by the optimizer: AdamW applies the weight decay
# directly to the weights, whereas an L2 penalty added to the loss would be
# distorted by Adam's adaptive steps
deep_classifier = MNISTClassifier(
  build_deep_model(), optimizer=torch.optim.AdamW, learning_rate=1e-3, weight_decay=0.01
)
print(ModelSummary(deep_classifier, max_depth=-1))

train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=128)
deep_trainer = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="deep"),
  # tqdm's progress bar, rather than rich's, which Lightning picks when rich
  # is installed (Colab): it froze some trainings
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
deep_trainer.fit(deep_classifier, train_loader, val_loader)
# The metrics of the last epoch, before the test replaces them
train_accuracy = float(deep_trainer.callback_metrics["train_accuracy"])
val_accuracy = float(deep_trainer.callback_metrics["val_accuracy"])
print(
  f"Accuracy of the last epoch: training {train_accuracy:.3f}, "
  f"validation {val_accuracy:.3f}"
)
evaluate(deep_trainer, deep_classifier)

# About 3 points more than adam_classifier (95.5% versus 92.5%), for 17,170
# parameters instead of 7,850: the gain of the architecture. Against the logistic
# regression trained with SGD, the 6 points of difference mix architecture and
# optimizer. At the last epoch, the training accuracy exceeds the validation one by one or two points:
# the beginning of overfitting, which the weight decay doesn't prevent on its own.

## Batch Size in TensorBoard

TensorBoard displays the learning curves that a `TensorBoardLogger`, passed to the `Trainer` through its `logger` argument, writes into a folder: one subfolder per training run. Run the following cell before the exercise: the dashboard fills up as the epochs go by (reload button at the top right). In a Codespace, if the dashboard doesn't show up under the cell, run `tensorboard --logdir logs` in a terminal, from the notebook's folder, and open the address it displays.

Then train two fresh multilayer perceptrons (`build_deep_model()`) with the settings of `deep_classifier` (`torch.optim.AdamW`, a learning rate of 10⁻³, a weight decay of 0.01, 10 epochs), one on batches of 128 examples, the other on batches of 1,500. Pass each `Trainer` a `TensorBoardLogger` writing into `logs/batch_128` for the first one, into `logs/batch_1500` for the second one.

Compare the `val_loss` and `val_accuracy` curves of the two training runs: which one converges faster, epoch by epoch? How many weight updates does each make per epoch? Why does the large batch converge more slowly per epoch?

**Useful documentation**

- [`TensorBoardLogger`](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.TensorBoardLogger.html)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `get_dataloaders(X_train, y_train, batch_size=batch_size)`: new `DataLoader`s for each batch size
- `TensorBoardLogger("logs", name="batch_128")`: the curves go into `logs/batch_128`
- A fresh model per run, `build_deep_model()`: otherwise the second run starts from the first one's weights
- In TensorBoard, the *Scalars* tab overlays the curves of the two folders; `CSVLogger` also writes into `logs`, but TensorBoard ignores its files
- Running a training again creates a new `version_<n>` subfolder: delete `logs` to start again from a clean dashboard
- To explain it, count the weight updates per epoch: the 48,000 training examples divided by the batch size

A skeleton to copy into the exercise cell, replacing each `...`:

```python
for batch_size in (128, 1500):
  train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=...)
  classifier = MNISTClassifier(
    build_deep_model(),
    optimizer=...,
    learning_rate=...,
    weight_decay=...,
  )
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("logs", name=f"batch_{...}"),
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifier, train_loader, val_loader)
```

### Solution

In [ ]:
for batch_size in (128, 1500):
  train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=batch_size)
  classifier = MNISTClassifier(
    build_deep_model(),
    optimizer=torch.optim.AdamW,
    learning_rate=1e-3,
    weight_decay=0.01,
  )
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("logs", name=f"batch_{batch_size}"),
    # tqdm's progress bar, rather than rich's, which Lightning picks when rich
    # is installed (Colab): it froze some trainings
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifier, train_loader, val_loader)
  val_accuracy = float(trainer.callback_metrics["val_accuracy"])
  print(f"Batches of {batch_size}: val_accuracy {val_accuracy:.3f}")

# Epoch by epoch, the small batches converge much faster: 92% validation accuracy
# from the first epoch and 95% at the tenth, while the batches of 1,500 start at
# 51% and end around 93%. Each epoch sees the same 48,000 examples, but makes 375
# weight updates with batches of 128 (48,000 / 128), and only 32 with batches of
# 1,500. The gradient of a large batch is less noisy, but at the same learning
# rate, an Adam step keeps roughly the same size: 12 times fewer steps cover less
# ground per epoch. An epoch of large batches is faster to compute, on the other
# hand, and a large batch withstands a larger learning rate.

## Hyperparameter Search

How many neurons in the hidden layer, which learning rate? [Optuna](https://optuna.org/) tries combinations of hyperparameters and keeps the best one, scored on the validation set. Its default sampler, TPE, draws its first 10 trials at random, then chooses each trial based on the results of the previous ones (slide "Going Further: Guided Search and Pruning").

Write an objective function, `objective(trial)`, which:

- draws the trial's hyperparameters: `units`, the number of hidden neurons, from 32 to 512 in steps of 32; `learning_rate`, Adam's learning rate, from 10⁻⁴ to 10⁻², on a logarithmic scale;
- trains a perceptron with one ReLU hidden layer and `nb_classes` output logits, wrapped in an `MNISTClassifier` with the `torch.optim.Adam` optimizer, on the integer labels: 3 epochs, batches of 1,500 examples, 20% validation;
- returns its final validation accuracy.

Then create a study maximizing this objective, run it for 20 trials, and display the table of the trials, the best hyperparameters and the best accuracy, and the history of the search (`plot_optimization_history`). Do the guided trials, after the first 10, do better than the random draws? Compare the best accuracy with the validation accuracy of `deep_classifier`, then compare the trials with each other: which of the two hyperparameters matters most, and why?

This is the longest part: a few minutes on CPU.

**Useful documentation**

- [`optuna.trial.Trial`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html)
- [`optuna.create_study`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.create_study.html)
- [`optuna.study.Study.optimize`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.Study.html#optuna.study.Study.optimize)
- [`optuna.visualization.plot_optimization_history`](https://optuna.readthedocs.io/en/stable/reference/visualization/generated/optuna.visualization.plot_optimization_history.html)

In [ ]:
def objective(trial: optuna.Trial) -> float: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `trial.suggest_int("units", 32, 512, step=32)` and `trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)` return the value of the current trial
- `nn.Sequential(nn.Linear(input_dim, units), nn.ReLU(), nn.Linear(units, nb_classes))`
- `lightning.Trainer(..., logger=False, enable_progress_bar=False, enable_model_summary=False)`: a silent training run, so that the 20 runs don't fill the output
- `trainer.callback_metrics["val_accuracy"]`: the validation accuracy of the last epoch, a tensor
- `optuna.create_study(direction="maximize")`: a study looking for the maximum of the objective; `study.optimize(objective, n_trials=20)` runs it
- `study.trials_dataframe()`, `study.best_params` and `study.best_value`: the trials, the best hyperparameters and the best value of the objective
- `optuna.visualization.plot_optimization_history(study).show()`: the accuracy of each trial, and the best one obtained so far
- To explain it, count the weight updates of each training: the 48,000 training examples divided by the batch size, times the number of epochs

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def objective(trial: optuna.Trial) -> float:
  units = trial.suggest_int("units", ...)
  learning_rate = trial.suggest_float("learning_rate", ...)
  model = nn.Sequential(
    nn.Linear(input_dim, ...), nn.ReLU(), nn.Linear(..., nb_classes)
  )
  classifier = MNISTClassifier(model, optimizer=..., learning_rate=...)
  train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=...)
  trainer = lightning.Trainer(
    max_epochs=...,
    accelerator="auto",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
  )
  trainer.fit(...)
  return float(trainer.callback_metrics[...])


study = optuna.create_study(direction=...)
study.optimize(objective, n_trials=...)
print(study.trials_dataframe())
print(study.best_params, study.best_value)
optuna.visualization.plot_optimization_history(...).show()
```

### Solution

In [ ]:
def objective(trial: optuna.Trial) -> float:
  units = trial.suggest_int("units", 32, 512, step=32)
  learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)

  model = nn.Sequential(
    nn.Linear(input_dim, units), nn.ReLU(), nn.Linear(units, nb_classes)
  )
  classifier = MNISTClassifier(
    model, optimizer=torch.optim.Adam, learning_rate=learning_rate
  )
  train_loader, val_loader = get_dataloaders(X_train, y_train, batch_size=1500)
  # A short training run, so that the search has time for 20 trials
  trainer = lightning.Trainer(
    max_epochs=3,
    accelerator="auto",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
  )
  trainer.fit(classifier, train_loader, val_loader)
  return float(trainer.callback_metrics["val_accuracy"])


study = optuna.create_study(direction="maximize", study_name="mnist")

In [ ]:
study.optimize(objective, n_trials=20)

In [ ]:
print(
  study.trials_dataframe()[["number", "value", "params_units", "params_learning_rate"]]
)
print(f"Best hyperparameters: {study.best_params}")
print(f"Best validation accuracy: {study.best_value:.4f}")
optuna.visualization.plot_optimization_history(study).show()

The learning rate matters more than the number of neurons. In one run, the trials with a learning rate close to 10⁻⁴ stay below 85% validation accuracy, whatever their size, while all those above 4·10⁻³ reach 95%. With batches of 1,500 examples, 3 epochs only make 96 weight updates (48,000 / 1,500 × 3), 12 times fewer per epoch than with batches of 128, as in the TensorBoard part: in so few steps, only a large learning rate converges. The search therefore ranks the configurations for this training budget, not for what they would give if trained longer.

The history shows TPE at work. Its first 10 trials are drawn at random (the `n_startup_trials` parameter of `optuna.samplers.TPESampler`), until it has gathered enough results to model the score, and range here from 69% to 95%. The next ones focus on large learning rates, and all of them reach 95%.

Compared with `deep_classifier` (about 95% validation accuracy, in 3,750 weight updates), the best trial does better in 96 updates: 97.2% in this run (512 neurons, a learning rate of 8.3·10⁻³), but with 24 times more parameters (407,050).